# Step 3: baseline training on Colab

Runtime > Change runtime type > **GPU** (T4 is enough). Run the cells top to bottom.
After a disconnect: rerun cells 1-5, skip the smoke test and the full run, and run **Resume**.

Weights and the MLflow DB are saved to Google Drive (`MyDrive/aerial-edge`); the dataset and the
live SQLite DB stay on the VM disk, which is faster and safer for SQLite.

In [ ]:
# 1. GPU check
!nvidia-smi

In [ ]:
# 2. Mount Google Drive (weights and the MLflow backup go here)
from google.colab import drive

drive.mount("/content/drive")
D = "/content/drive/MyDrive/aerial-edge"
!mkdir -p {D}

In [ ]:
# 3. Get the code. For a private repo use https://<token>@github.com/...
REPO_URL = "https://github.com/mttrin93/aerial-edge.git"  # change to your repo
REPO = "/content/aerial-edge"
!test -d {REPO} || git clone {REPO_URL} {REPO}
%cd {REPO}
!git pull --ff-only

In [ ]:
# 4. Install: keep Colab's CUDA torch, add only what training needs.
# PYTHONPATH instead of `pip install -e .`, so the package's Python pin does not matter here.
!pip install -q ultralytics==8.4.163 mlflow==3.16.1
%env PYTHONPATH=/content/aerial-edge/src
%env MLFLOW_TRACKING_URI=sqlite:////content/mlflow.db

In [ ]:
# 5. Data: VisDrone2019-DET (~2 GB) to the VM disk, a few minutes per session
!yolo settings datasets_dir=/content/datasets
!python scripts/download_data.py

## Smoke test

One epoch on 1% of the data, logged to a separate throwaway DB and folder. Checks that data,
GPU, MLflow and the Drive backup all work before the long run.

In [ ]:
!MLFLOW_TRACKING_URI=sqlite:////content/smoke.db python scripts/train.py epochs=1 fraction=0.01 \
    project=/content/smoke --backup-mlflow /content/smoke.db {D}/smoke.db

## Full run

100 epochs, settings from `configs/train.yaml`. Weights go to `{D}/runs/yolo26n_640/weights`.

In [ ]:
!python scripts/train.py project={D}/runs --backup-mlflow /content/mlflow.db {D}/mlflow.db

## Resume (only after a disconnect)

Restore the MLflow DB from Drive, then continue from `last.pt`.

In [ ]:
!cp {D}/mlflow.db /content/mlflow.db
!python scripts/train.py --resume {D}/runs/yolo26n_640/weights/last.pt \
    --backup-mlflow /content/mlflow.db {D}/mlflow.db

## Results

Last epochs from `results.csv`. Download `best.pt` and `mlflow.db` from Drive to the laptop:
`best.pt` into `models/`, `mlflow.db` into `mlflow/`.

In [ ]:
import pandas as pd

df = pd.read_csv(f"{D}/runs/yolo26n_640/results.csv")
df[["epoch", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail()